# 04 — Flow Architecture

lionag2 uses a **Flow-native architecture**:
- **Flow**: a shared `Pile` (ID-keyed collection) + named `Progression` streams.
- **Per-agent streams**: each agent gets its own named `MemoryStream` backed by `FlowStorage`.
- **Bridge observers**: `@agent.observer(EventType)` on each agent forwards typed events to the engine for reactive coordination.

All coordination state lives in the Flow as typed events — the entire execution is replayable from `flow.to_dict()`.

In [1]:
from lionag2.core import Flow
from lionag2.research.events import FindingEmitted

## Flow — shared state

The `Flow` owns a shared `Pile` (all events) and named streams (one per agent). Each stream is a `MemoryStream` backed by `FlowStorage` — an AG2 `Storage` protocol adapter.

In [2]:
flow = Flow(name="research")

# Named streams are created on access
team_a = flow.streams["team_d0_abc123"]
team_b = flow.streams["team_d1_def456"]

print(f"Flow: {flow}")
print(f"Streams: {list(flow.progression_names)}")
print(f"Shared pile: {flow.items}")

Flow: Flow(items=0, progressions=[team_d0_abc123:0, team_d1_def456:0])
Streams: ['team_d0_abc123', 'team_d1_def456']
Shared pile: Pile(len=0, types={})


## Bridge observers in action

When an agent emits a `FindingEmitted`, a bridge observer on the agent forwards it to the engine. The engine records it in the Flow and decides whether to spawn a child team.

In [3]:
# Events land in the Flow's shared Pile via FlowStorage

finding = FindingEmitted(
    claim="Cuprate Tc peaks at 16% hole doping",
    evidence="Phase diagram data",
    source_agent="surveyor",
    novelty=0.8,
    depth=0,
)

# In the engine, this happens via agent observer → engine._record()
flow.items.include(finding)

print(f"Flow items: {len(flow.items)}")
print(f"Findings: {len(flow.items.by_type(FindingEmitted))}")
if flow.items.by_type(FindingEmitted):
    print(f"  claim: {flow.items.by_type(FindingEmitted)[0].claim}")

Flow items: 1
Findings: 1
  claim: Cuprate Tc peaks at 16% hole doping


## Querying the Flow

The Flow's Pile supports rich querying via AG2's Condition DSL:

In [4]:
# Type-based lookup
all_findings = flow.items.by_type(FindingEmitted)
print(f"All findings: {len(all_findings)}")

# Condition DSL (if AG2 Condition is available)
# high_novelty = flow.items[FindingEmitted.novelty > 0.7]

# Progression order (per-stream)
for name in flow.progression_names:
    events = flow[name]
    print(f"  {name}: {len(events)} events")

All findings: 1
  team_d0_abc123: 0 events
  team_d1_def456: 0 events


## Depth-aware context carry-over

When a child team spawns at depth+1, it receives parent findings as context. The engine filters by `node_id` to scope findings to the specific parent branch:

```python
parent_findings = [
    f for f in flow.items.by_type(FindingEmitted)
    if f.node_id == parent_node_id
]
```

This prevents context dilution — each child sees only its parent's discoveries, not the entire tree.

In [5]:
# Serialization round-trip
import json

data = flow.to_dict()
print(f"Serialized: {len(json.dumps(data, default=str))} bytes")

restored = Flow.from_dict(data)
print(f"Restored: {len(restored.items)} items, {len(restored.progression_names)} streams")

Serialized: 405 bytes
Restored: 1 items, 2 streams


## Up next

Events land in the Flow. Now we need something to **react** to them. Tutorial 05 introduces bridge observers — the reactive handlers that turn findings into depth expansions.